In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/geoap96/deepglobe2018-landcover-segmentation-traindataset/data.zip


In [2]:
import os

BASE = "/kaggle/input/datasets/geoap96/deepglobe2018-landcover-segmentation-traindataset"
print("Level 1:", os.listdir(BASE))

Level 1: ['data.zip']


In [3]:
import zipfile
import os

ZIP_PATH = "/kaggle/input/datasets/geoap96/deepglobe2018-landcover-segmentation-traindataset/data.zip"
EXTRACT_PATH = "/kaggle/working/data"
if not os.path.exists(EXTRACT_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Extracted")
else:
    print("Already extracted")

Extracted


In [4]:
import os

BASE = "/kaggle/working/data"
for root, dirs, files in os.walk(BASE):
    print(root)

/kaggle/working/data
/kaggle/working/data/data
/kaggle/working/data/data/training_data
/kaggle/working/data/data/training_data/images
/kaggle/working/data/data/training_data/masks
/kaggle/working/data/data/test_data
/kaggle/working/data/data/test_data/images
/kaggle/working/data/data/test_data/masks


In [5]:
IMAGE_DIR = "/kaggle/working/data/data/training_data/images"
MASK_DIR = "/kaggle/working/data/data/training_data/masks"
print("Images:", len(os.listdir(IMAGE_DIR)))
print("Masks:", len(os.listdir(MASK_DIR)))

Images: 683
Masks: 683


In [8]:
import cv2
import torch
from torch.utils.data import Dataset
import os

class LandDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        name = self.images[idx]
        img_path = os.path.join(self.img_dir, name)
        mask_name = name.replace("_sat.jpg", "_mask.png")
        mask_path = os.path.join(self.mask_dir, mask_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, 0)
        img = cv2.resize(img, (256, 256))
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
        img = img / 255.0
        img = torch.tensor(img).permute(2, 0, 1).float()
        mask = torch.tensor(mask).long()

        return img, mask

In [9]:
from torch.utils.data import DataLoader
import torch

dataset = LandDataset(IMAGE_DIR, MASK_DIR)
loader = DataLoader(dataset, batch_size=8, shuffle=True)
print("Dataset size:", len(dataset))
img, mask = dataset[0]
print("Image shape:", img.shape)
print("Mask unique values:", torch.unique(mask))

Dataset size: 683
Image shape: torch.Size([3, 256, 256])
Mask unique values: tensor([ 29, 178, 225])


In [10]:
def encode_mask(mask):
    import numpy as np
    new_mask = np.zeros_like(mask)
    new_mask[mask == 29] = 0      # urban
    new_mask[mask == 76] = 1      # agriculture
    new_mask[mask == 150] = 2     # rangeland
    new_mask[mask == 178] = 3     # forest
    new_mask[mask == 225] = 4     # water
    new_mask[mask == 255] = 5     # barren

    return new_mask

In [11]:
class LandDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        name = self.images[idx]
        img_path = os.path.join(self.img_dir, name)
        mask_name = name.replace("_sat.jpg", "_mask.png")
        mask_path = os.path.join(self.mask_dir, mask_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, 0)
        mask = encode_mask(mask)
        img = cv2.resize(img, (256, 256))
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
        img = img / 255.0
        img = torch.tensor(img).permute(2, 0, 1).float()
        mask = torch.tensor(mask).long()

        return img, mask

In [12]:
dataset = LandDataset(IMAGE_DIR, MASK_DIR)
img, mask = dataset[0]
import torch
print("Mask unique values:", torch.unique(mask))

Mask unique values: tensor([0, 3, 4])


In [14]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.4 MB/s eta 0:00:00


In [15]:
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch

model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=6
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print("Model ready on:", device)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Model ready on: cuda


In [16]:
from torch.utils.data import DataLoader

dataset = LandDataset(IMAGE_DIR, MASK_DIR)
loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("DataLoader ready")

DataLoader ready


In [17]:
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
import os

scaler = GradScaler()

CHECKPOINT_PATH = "/kaggle/working/checkpoint.pth"
BEST_MODEL_PATH = "/kaggle/working/best_model.pth"

start_epoch = 0
best_loss = float("inf")

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    start_epoch = checkpoint["epoch"]
    best_loss = checkpoint["best_loss"]
    print("Resuming from epoch:", start_epoch)

EPOCHS = 25

for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(loader)

    for imgs, masks in loop:
        imgs = imgs.to(device)
        masks = masks.to(device)

        with autocast():
            preds = model(imgs)
            loss = criterion(preds, masks)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch} Avg Loss: {avg_loss}")
    torch.save({
        "epoch": epoch + 1,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_loss": best_loss
    }, CHECKPOINT_PATH)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Best model saved!")

/tmp/ipykernel_55/3120929002.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
  0%|          | 0/86 [00:00<?, ?it/s]/tmp/ipykernel_55/3120929002.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 0: 100%|██████████| 86/86 [00:47<00:00,  1.83it/s, loss=0.739]


Epoch 0 Avg Loss: 1.0431948837845824
Best model saved!


Epoch 1: 100%|██████████| 86/86 [00:46<00:00,  1.84it/s, loss=0.853]


Epoch 1 Avg Loss: 0.7210884267507598
Best model saved!


Epoch 2: 100%|██████████| 86/86 [00:46<00:00,  1.83it/s, loss=0.905]


...

Epoch 22 Avg Loss: 0.18532204030211583
Best model saved!


Epoch 23: 100%|██████████| 86/86 [00:48<00:00,  1.78it/s, loss=0.22]  


Epoch 23 Avg Loss: 0.1817636553977811
Best model saved!


Epoch 24: 100%|██████████| 86/86 [00:48<00:00,  1.78it/s, loss=0.167] 


Epoch 24 Avg Loss: 0.17729625876906308
Best model saved!
